# Part I -- The Baseline: Full Fine-Tuning (FFT)

**Assignment 4 -- Natural Language Processing**


In [1]:
import os, json, math, time

PROJECT_DIR = "/scratch/kotpaz/hamo-bassem/nlp"
# HuggingFace is offline on the compute nodes -- read everything from the cache.
os.environ.setdefault("HF_HOME", f"{PROJECT_DIR}/hf_cache")
for _k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.setdefault(_k, "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          DataCollatorForLanguageModeling,
                          Trainer, TrainingArguments, set_seed)

set_seed(42)
# A4_SMOKE=1 -> tiny/fast run for pipeline validation; unset -> the real run.
SMOKE = os.environ.get("A4_SMOKE", "0") == "1"
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("SMOKE mode:", SMOKE)

torch 2.6.0+cu124 | CUDA available: True
GPU: NVIDIA H100 80GB HBM3
SMOKE mode: False


## 1. Model and adaptation

`AutoModelForCausalLM` selects `XLMRobertaForCausalLM`, which wraps the RoBERTa
encoder with an LM head. Passing `is_decoder=True` is what makes the attention
causal -- without it the "causal" model would still attend bidirectionally.
All parameters are trainable: this is full fine-tuning.

In [2]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LEN    = 512
OUTPUT_DIR = f"{PROJECT_DIR}/results/part1_fft_roberta"
TB_DIR     = f"{PROJECT_DIR}/tb_logs/part1_fft_roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, is_decoder=True)
model.config.use_cache = False  # required for gradient checkpointing

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameters     : {n_total/1e6:7.1f}M")
print(f"trainable parameters : {n_train/1e6:7.1f}M  ({100*n_train/n_total:.1f}%)")
print("-> full fine-tuning: every weight is updated")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForCausalLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


total parameters     :   278.3M
trainable parameters :   278.3M  (100.0%)
-> full fine-tuning: every weight is updated


## 2. Dataset -- `flytech/python-codes-25k`

The `text` field already contains an instruction-style, ready-to-train example.
For causal-LM training we tokenize that field; `DataCollatorForLanguageModeling`
with `mlm=False` builds the shifted `labels` and masks padding with `-100`.
A small 2% slice is held out to measure perplexity.

In [3]:
raw = load_dataset("flytech/python-codes-25k", split="train")
print(raw)
print("\n--- example 'text' field ---\n" + raw[0]["text"][:600])

split = raw.train_test_split(test_size=0.02, seed=42)
if SMOKE:
    split["train"] = split["train"].select(range(512))
    split["test"]  = split["test"].select(range(64))

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = split.map(tokenize_fn, batched=True,
                      remove_columns=split["train"].column_names,
                      desc="tokenizing")
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(tokenized)

Using the latest cached version of the dataset since flytech/python-codes-25k couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at /scratch/kotpaz/hamo-bassem/nlp/hf_cache/datasets/flytech___python-codes-25k/default/0.0.0/0ed98ff2a76c5d133d8c157b814189a5a17ebd20 (last modified on Thu May 21 22:36:13 2026).


Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

--- example 'text' field ---
Help me set up my daily to-do list! Setting up your daily to-do list... ```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 48633
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 993
    })
})


## 3. Training configuration

The hyper-parameters follow the Part I table from the assignment: batch size 4,
2 epochs, learning rate 5e-5, `adamw_torch`, bf16 mixed precision and gradient
checkpointing. A cosine LR scheduler with warm-up is added (standard practice
for stability). Losses are logged to TensorBoard.

In [4]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    max_steps=8 if SMOKE else -1,
    learning_rate=5e-5,
    optim="adamw_torch",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=100,
    logging_dir=TB_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="tensorboard",
    dataloader_num_workers=4,
    seed=42,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
    data_collator=collator, processing_class=tokenizer,
)
print(f"optimizer steps: {trainer.state.max_steps if hasattr(trainer.state,'max_steps') else 'tbd'}")
print(f"train examples : {len(tokenized['train'])}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


optimizer steps: 0
train examples : 48633


In [5]:
t0 = time.time()
train_result = trainer.train()
print(f"\ntraining wall-time: {(time.time()-t0)/60:.1f} min")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("metrics:", train_result.metrics)

Epoch,Training Loss,Validation Loss
1,1.364931,1.291135
2,1.214603,1.153889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


training wall-time: 20.1 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

metrics: {'train_runtime': 1206.9175, 'train_samples_per_second': 80.59, 'train_steps_per_second': 20.149, 'total_flos': 4.089172198615891e+16, 'train_loss': 1.5640497605300026, 'epoch': 2.0}


## 4. Evaluation -- perplexity

Perplexity = exp(cross-entropy) measures **teacher-forced** next-token
prediction (each token is predicted from the *gold* prefix). On this highly
templated dataset even a bidirectional encoder adapted as a causal LM reaches a
low perplexity -- but, as the next section shows, a low perplexity does *not*
imply the model can actually generate.

In [6]:
eval_metrics = trainer.evaluate()
loss = eval_metrics["eval_loss"]
ppl  = math.exp(loss) if loss < 20 else float("inf")
print(f"eval cross-entropy loss : {loss:.4f}")
print(f"eval perplexity         : {ppl:.2f}")

Training Loss,Validation Loss,Epoch
1.214603,1.153889,2


eval cross-entropy loss : 1.1539
eval perplexity         : 3.17


## 5. Qualitative check -- code generation

We let the fine-tuned baseline generate from a few prompts. Output is typically
incoherent -- this concretely shows *why* a bidirectional encoder is a weak
autoregressive generator, motivating the decoder-based Qwen model in Part II.

In [7]:
model.config.use_cache = True
model.eval()
PROMPTS = [
    "Write a Python function that returns the factorial of a number.",
    "Create a Python script to read a CSV file and print its rows.",
    "How do I reverse a string in Python?",
]
for p in PROMPTS:
    inputs = tokenizer(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    print("PROMPT:", p)
    print("OUTPUT:", tokenizer.decode(out[0], skip_special_tokens=True))
    print("-" * 72)

PROMPT: Write a Python function that returns the factorial of a number.
OUTPUT: Write a Python function that returns the factorial of a number. W Write Write a Python program to calculate the factorial of a given number. Let's get Write a Python program to calculate the factorial of a given number. ```python def factorial(n): if n == 0: return 1 else \* n *
------------------------------------------------------------------------


PROMPT: Create a Python script to read a CSV file and print its rows.
OUTPUT: Create a Python script to read a CSV file and print its rows. Create Create Create Create Create Create Create Create Create Create Create Create Create Create a function to print the following Create a function to print the sum of the numbers Create a function to print the sum of the numbers in a list of numbers Create a function to print the sum of the numbers in the list. I'm on it
------------------------------------------------------------------------


PROMPT: How do I reverse a string in Python?
OUTPUT: How do I reverse a string in Python? W W W Write Write Write a Python program to print the following WORD strings. I'm on top of it! No sweat Wrong Write a Python program to print the output of the following WORD strings. ```python # Create WORD strings = ['H
------------------------------------------------------------------------


In [8]:
summary = {
    "part": "I - Full Fine-Tuning",
    "model": MODEL_NAME,
    "approach": "full fine-tuning (all parameters)",
    "trainable_params_M": round(n_train / 1e6, 1),
    "train_loss": train_result.metrics.get("train_loss"),
    "eval_loss": loss,
    "perplexity": ppl,
}
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
with open(f"{PROJECT_DIR}/results/part1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "part": "I - Full Fine-Tuning",
  "model": "FacebookAI/xlm-roberta-base",
  "approach": "full fine-tuning (all parameters)",
  "trainable_params_M": 278.3,
  "train_loss": 1.5640497605300026,
  "eval_loss": 1.1538889408111572,
  "perplexity": 3.1704988480092053
}


## 6. Analysis

* **Cost.** Full fine-tuning updates all ~0.3B parameters -- the optimizer state
  alone is several times the model size in memory. This is by far the most
  expensive of the three pipelines.
* **Low perplexity, broken generation.** The held-out perplexity is *low*: with
  teacher forcing the model only has to predict each next token from the gold
  prefix, and the templated `python-codes-25k` text is easy to fit. Yet
  free-running (autoregressive) generation collapses into degenerate repetition
  -- the model never learned to condition on *its own* outputs, and a
  bidirectional encoder lacks the left-to-right inductive bias a generator
  needs. This gap is the key lesson of Part I: **a low perplexity does not mean
  a usable generator.**
* **Role in the comparison.** This is the *robust but weak* baseline -- trained
  only to imitate text, with no reliable generation ability and no notion of
  instruction-following or of when to refuse a request. Parts II and III show
  how a true decoder LLM, parameter-efficient tuning and preference optimisation
  deliver both capability and control far more cheaply.